<a id="spherical-surface"></a>

$$
\text{几何光学 (Geometric Optics): 球面折射成像} \\ \quad \\
\text{傍轴近似 (Paraxial Approximation): 光线和光轴的夹角很小, 用泰勒展开的一阶项} \\
\sin\theta = \theta - \frac{\theta^3}{6} + \cdots \approx \theta, \quad \tan\theta \approx \theta, \quad \cos\theta \approx 1 \\
\text{斯涅尔定律变成线性的: } n_1\theta_1 = n_2\theta_2 \\ \quad \\
\text{单个球面 (半径 R, 球心在右侧为正), 左边折射率 } n_1 \text{, 右边 } n_2 \\
\text{物点 O 在左边距离 } s_o \text{ 处, 一条光线在高度 y 处打到球面, 折射后交光轴于像点 I (右边距离 } s_i \text{)} \\
\text{三个角 (都是和光轴的夹角, 傍轴): } \alpha = \frac{y}{s_o} \ (\text{入射光线}), \ \beta = \frac{y}{s_i} \ (\text{折射光线}), \ \varphi = \frac{y}{R} \ (\text{法线, 即球心连线}) \\
\text{入射角 (和法线的夹角) } \theta_1 = \alpha + \varphi \text{, 折射角 } \theta_2 = \varphi - \beta \quad (\text{三角形外角定理}) \\
n_1(\alpha + \varphi) = n_2(\varphi - \beta) \implies n_1\frac{y}{s_o} + n_2\frac{y}{s_i} = (n_2 - n_1)\frac{y}{R} \\
\frac{n_1}{s_o} + \frac{n_2}{s_i} = \frac{n_2 - n_1}{R} \\
\text{y 约掉了: 傍轴近似下, 从 O 出发的所有光线都汇聚到同一个点 I (理想成像)} \\
\text{不做傍轴近似时, 不同高度的光线汇聚点不同 → 球差 (Spherical Aberration)}
$$

数学基础: [泰勒展开](../math_basics/calculus.ipynb#taylor) · 应用: [斯涅尔定律](reflection_refraction.ipynb#fermat-snell)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

n1, n2, R = 1.0, 1.5, 10.0
s_o = 40.0
s_i_paraxial = n2 / ((n2 - n1) / R - n1 / s_o)
print(f'球面成像公式: s_i = {s_i_paraxial:.6f}')

def exact_trace(y):
    # 精确光线追迹: 物点 (-s_o, 0) 射向球面上高度 y 的点, 用真正的斯涅尔定律
    z = R - np.sqrt(R**2 - y**2)                  # 球面顶点在 z = 0, 球心在 (R, 0)
    d_in = np.array([z + s_o, y]); d_in /= np.linalg.norm(d_in)
    normal = np.array([z - R, y]); normal /= np.linalg.norm(normal)   # 从球心指向交点, 朝向介质 1
    cos_i = -d_in @ normal
    eta = n1 / n2
    cos_t = np.sqrt(1 - eta**2 * (1 - cos_i**2))
    d_out = eta * d_in + (eta * cos_i - cos_t) * normal       # 矢量形式的斯涅尔定律
    t = -y / d_out[1]                                          # 和光轴 (y = 0) 的交点
    return z + t * d_out[0]

for y in [0.01, 0.5, 2.0, 5.0]:
    print(f'光线高度 y = {y:<5}: 精确追迹交点 = {exact_trace(y):.6f} (傍轴 {s_i_paraxial:.6f}), 偏差 = {exact_trace(y) - s_i_paraxial:+.4f}')
print('y 越大偏差越大: 球差')

<a id="thin-lens"></a>

$$
\text{薄透镜公式 (Thin Lens Equation)} \\
\text{透镜 = 两个球面 (半径 } R_1, R_2 \text{), 玻璃折射率 n, 放在空气中, 厚度忽略} \\
\text{第一个面 (空气→玻璃): } \frac{1}{s_o} + \frac{n}{s_1} = \frac{n - 1}{R_1} \\
\text{第一个面的像作为第二个面的物, 在第二个面的右边 } s_1 \text{ 处 (虚物 virtual object), 所以物距 = } -s_1 \\
\text{第二个面 (玻璃→空气): } \frac{n}{-s_1} + \frac{1}{s_i} = \frac{1 - n}{R_2} \\
\text{两式相加, } \frac{n}{s_1} \text{ 抵消:} \\
\frac{1}{s_o} + \frac{1}{s_i} = (n - 1)\left(\frac{1}{R_1} - \frac{1}{R_2}\right) = \frac{1}{f} \quad (\text{磨镜者公式 Lensmaker's Equation}) \\ \quad \\
\text{焦距 (Focal Length) f: 平行光 } (s_o \to \infty) \text{ 汇聚的位置 } s_i = f \\
\text{放大率 (Magnification): 通过透镜中心的光线不偏折, 相似三角形 } \implies m = \frac{h_i}{h_o} = -\frac{s_i}{s_o} \text{ (负号: 倒立)}
$$

<a id="abcd"></a>

$$
\text{光线传输矩阵 (Ray Transfer Matrix / ABCD Matrix): 傍轴光学是线性的} \\
\text{每条光线用一个向量表示 } \begin{bmatrix} y \\ \theta \end{bmatrix} \text{ (高度, 和光轴的夹角), 每个光学元件是一个 2x2 矩阵} \\ \quad \\
\text{1. 自由传播距离 d: 高度增加 } d\tan\theta \approx d\theta \text{, 角度不变} \\
\begin{bmatrix} y' \\ \theta' \end{bmatrix} = \begin{bmatrix} 1 & d \\ 0 & 1 \end{bmatrix}\begin{bmatrix} y \\ \theta \end{bmatrix} \\
\text{2. 薄透镜: 高度不变; 平行光 } (\theta = 0) \text{ 在高度 y 处折射后指向焦点, } \theta' = -\frac{y}{f} \text{; 由线性叠加} \\
\begin{bmatrix} y' \\ \theta' \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ -\frac{1}{f} & 1 \end{bmatrix}\begin{bmatrix} y \\ \theta \end{bmatrix} \\
\text{3. 球面折射: 由上面的推导 } n_2(\varphi - \beta') = n_1(\alpha + \varphi) \text{, 用和光轴的有向夹角 } \theta = \alpha, \ \theta' = -\beta' \text{:} \\
\theta' = \frac{n_1}{n_2}\theta - \frac{n_2 - n_1}{n_2 R} y \implies \begin{bmatrix} 1 & 0 \\ -\frac{n_2 - n_1}{n_2 R} & \frac{n_1}{n_2} \end{bmatrix} \\
\text{4. 球面镜 (半径 R): } \begin{bmatrix} 1 & 0 \\ -\frac{2}{R} & 1 \end{bmatrix} \quad (f = \frac{R}{2}) \\ \quad \\
\text{整个系统的矩阵 = 按光线经过的顺序从右往左相乘: } M = M_k \cdots M_2 M_1 \\
\det M = \prod \det M_i \text{, 传播和透镜的行列式都是 1, 球面折射是 } \frac{n_1}{n_2} \implies \det M = \frac{n_{\text{入}}}{n_{\text{出}}} \text{ (同一种介质中为 1)} \\ \quad \\
\text{成像条件: } M = P(d_i) \, L(f) \, P(d_o) = \begin{bmatrix} 1 - \frac{d_i}{f} & d_o + d_i - \frac{d_o d_i}{f} \\ -\frac{1}{f} & 1 - \frac{d_o}{f} \end{bmatrix} \\
\text{成像 = 从同一个物点出发, 不同角度的光线到达同一个像点 } \iff y' = A y + B\theta \text{ 和 } \theta \text{ 无关} \iff B = 0 \\
d_o + d_i - \frac{d_o d_i}{f} = 0 \xrightarrow{\text{除以 } d_o d_i} \frac{1}{d_i} + \frac{1}{d_o} = \frac{1}{f} \quad (\text{就是薄透镜公式}) \\
\text{此时放大率 } m = A = 1 - \frac{d_i}{f} = 1 - d_i\left(\frac{1}{d_o} + \frac{1}{d_i}\right) = -\frac{d_i}{d_o} \\ \quad \\
\text{两个透镜相距 d: } M = L(f_2) \, P(d) \, L(f_1) \text{, 它的 C 元素} = -\frac{1}{f_{\text{eff}}} \\
\frac{1}{f_{\text{eff}}} = \frac{1}{f_1} + \frac{1}{f_2} - \frac{d}{f_1 f_2} \quad (d = 0 \text{ 时焦度 (power) 直接相加})
$$

数学基础: [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication) · [行列式](../math_basics/linear_algebra.ipynb#determinant) · [泰勒展开](../math_basics/calculus.ipynb#taylor)

In [ ]:
def prop(d):
    return np.array([[1.0, d], [0.0, 1.0]])

def lens(f):
    return np.array([[1.0, 0.0], [-1.0 / f, 1.0]])

def surface(n1, n2, R):
    return np.array([[1.0, 0.0], [-(n2 - n1) / (n2 * R), n1 / n2]])

def mirror(R):
    return np.array([[1.0, 0.0], [-2.0 / R, 1.0]])

# 1. 球面矩阵 vs 球面成像公式
M = prop(s_i_paraxial) @ surface(n1, n2, R) @ prop(s_o)
print(f'单球面: B = {M[0, 1]:.2e} (成像), det = {np.linalg.det(M):.6f} == n1 / n2 = {n1 / n2:.6f}')

# 2. 两个球面组成薄透镜 (中间厚度 0) == 磨镜者公式
n_glass, R1, R2 = 1.5, 20.0, -30.0
M_two = surface(n_glass, 1.0, R2) @ surface(1.0, n_glass, R1)
f_lensmaker = 1 / ((n_glass - 1) * (1 / R1 - 1 / R2))
print(f'两个球面相乘 = \n{M_two}\n薄透镜矩阵 (f = {f_lensmaker:.4f}) = \n{lens(f_lensmaker)}\n一致? {np.allclose(M_two, lens(f_lensmaker))}')

# 3. 成像: B = 0 <=> 薄透镜公式, A = 放大率
f = 10.0
for d_o in [15.0, 30.0, 60.0]:
    d_i = 1 / (1 / f - 1 / d_o)
    M = prop(d_i) @ lens(f) @ prop(d_o)
    print(f'd_o = {d_o}: d_i = {d_i:.4f}, B = {M[0, 1]:.1e}, A = {M[0, 0]:.4f}, -d_i/d_o = {-d_i / d_o:.4f}, det = {np.linalg.det(M):.4f}')

# 4. 两个透镜的等效焦距
f1, f2, d = 10.0, 20.0, 5.0
M_sys = lens(f2) @ prop(d) @ lens(f1)
print(f'两个透镜: -1 / C = {-1 / M_sys[1, 0]:.6f}, 公式 f_eff = {1 / (1 / f1 + 1 / f2 - d / (f1 * f2)):.6f}')

# 5. 球面镜焦距 R / 2: 平行光经过镜面后在 R/2 处过光轴
R_m = 40.0
for y0 in [1.0, 3.0]:
    out = prop(R_m / 2) @ mirror(R_m) @ np.array([y0, 0.0])
    print(f'球面镜: 高度 {y0} 的平行光, 传播 R/2 后高度 = {out[0]:.2e}')

In [ ]:
# 光线追迹图: 从物点出发的多条光线, 经过透镜后汇聚到像点
f = 10.0
d_o, h_o = 25.0, 2.0
d_i = 1 / (1 / f - 1 / d_o)
h_i = -d_i / d_o * h_o

plt.figure(figsize=(8, 3))
for theta0 in np.linspace(-0.15, 0.05, 6):
    ray = np.array([h_o, theta0])
    z_pts, y_pts = [0.0], [ray[0]]
    ray = prop(d_o) @ ray
    z_pts.append(d_o); y_pts.append(ray[0])
    ray = lens(f) @ ray
    ray = prop(d_i) @ ray
    z_pts.append(d_o + d_i); y_pts.append(ray[0])
    plt.plot(z_pts, y_pts, 'r-', linewidth=0.8)
plt.annotate('', xy=(0, h_o), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='b', lw=2))
plt.annotate('', xy=(d_o + d_i, h_i), xytext=(d_o + d_i, 0), arrowprops=dict(arrowstyle='->', color='g', lw=2))
plt.axvline(d_o, color='k', linewidth=2, label='thin lens')
plt.plot([d_o - f, d_o + f], [0, 0], 'kx', label='focal points')
plt.axhline(0, color='gray', linewidth=0.5)
plt.title(f'd_o = {d_o}, f = {f}  ->  d_i = {d_i:.2f}, m = {-d_i / d_o:.2f}')
plt.xlabel('z')
plt.ylabel('y')
plot_nicely()
plt.show()
print(f'像高 = {h_i:.4f}, 所有光线在 z = {d_o + d_i:.4f} 处的高度相同 (B = 0)')